In [2]:
import os
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import gc
from tqdm.auto import tqdm
from joblib import Parallel, delayed
from utils.collection.tensors import import_npz
from utils.preprocessing.splitting import split_and_balance, task2_preprocess
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

cur_dir = Path(os.getcwd())
proj_dir = cur_dir.resolve().parent.parent
LABEL_DATA_PATH = proj_dir / "data" / "labels"
FEATURE_DATA_PATH = proj_dir / "data" / "features"
TASK_1_DATA_PATH = proj_dir / "data" / "split" / "task_1"
TASK_2_DATA_PATH = proj_dir / "data" / "split" / "task_2"

SELECTED_SUBJECTS = [4,12,13,19]
SELECTED_NAMES = [f"chb{str(pid).zfill(2)}" for pid in SELECTED_SUBJECTS]
SELECTED_NAMES

['chb04', 'chb12', 'chb13', 'chb19']

# Data Splitting

The system processes data extracted from the `features` and `labels` files to generate the datasets required for model training. For each processed file, the procedure follows these steps:

1.  **Chronological Splitting:** The dataset is divided in strict chronological order to respect the sequential nature of the EEG signal. This ensures the integrity of the **train**, **validation**, and **test** sets, preventing "look-ahead bias."
2.  **Class Balancing:** The training and validation sets are processed using a **random undersampling** technique. This step is essential to address the severe class imbalance between rare pre-ictal events and long periods of normal activity (inter-ictal).
3.  **Task 2 Filtering (Pre-ictal Prediction):** For the pre-seizure analysis, samples labeled as active seizures (ictal) are removed from the dataset. The target labels are then reassigned to focus on predicting the specific time window between **1 and 10 minutes** before the seizure onset.
4.  **Data Archiving:** The final results are saved in compressed `.npz` files to optimize disk space and loading speed. Each file contains the following keys:
    * **Train Set:** `X_train`, `y_train`, `t_train`
    * **Validation Set:** `X_val`, `y_val`, `t_val`
    * **Test Set:** `X_test`, `y_test`, `t_test` (maintained as imbalanced and chronological to allow for a realistic performance evaluation)
    * **Metadata:** `feature_names`, `channels`

In [3]:
for p_name in tqdm(SELECTED_NAMES, desc="Subjects"):    
    
    print(f"\nElaborating {p_name}")
    # define paths
    X_data_dir = FEATURE_DATA_PATH / p_name
    y_data_dir = LABEL_DATA_PATH / p_name
    task_1_dir = TASK_1_DATA_PATH / p_name
    task_2_dir = TASK_2_DATA_PATH / p_name
    task_1_dir.mkdir(parents=True, exist_ok=True)
    task_2_dir.mkdir(parents=True, exist_ok=True)
    
    # find files
    X_data_files = sorted(list(X_data_dir.glob("*.npz")))
    y_data_files = sorted(list(y_data_dir.glob("*.npz")))
    
    for x_path, y_path in tqdm(zip(X_data_files, y_data_files), total=len(X_data_files), desc=f"Labels ({p_name})", leave=False):
        
        # load file
        X_data = np.load(x_path)
        X = X_data["features"]
        feature_names = X_data["feature_names"]
        times = X_data['times']
        channels = X_data['channels']
        
        del X_data
        
        y_data = np.load(y_path)
        y_task_1 = y_data["y_task1"]
        y_task_2 = y_data["y_task2"]
        
        del y_data

        print("\n|--- Splitting task 1 ---|")
        train_1, val_1, test_1 = split_and_balance(X, y_task_1, times)
        
        # save Task 01 datasets
        np.savez(task_1_dir / x_path.name, 
                X_train=train_1[0], y_train=train_1[1], t_train=train_1[2],
                X_val=val_1[0], y_val=val_1[1], t_val=val_1[2],
                X_test=test_1[0], y_test=test_1[1], t_test=test_1[2],
                feature_names=feature_names,
                channels=channels
                )
        
        print("\n|-----------------------|")

        del train_1, val_1, test_1
        
        print("\n|--- Splitting task 2 ---|")

        X_task_2, y_task_2, times_2 = task2_preprocess(X, y_task_1, y_task_2, times)
        
        del X, y_task_1, times

        train_2, val_2, test_2= split_and_balance(X_task_2, y_task_2, times_2)
        
        del X_task_2, y_task_2, times_2
        
        # save Task 02 datasets
        np.savez(task_2_dir / x_path.name, 
                X_train=train_2[0], y_train=train_2[1], t_train=train_2[2],
                X_val=val_2[0], y_val=val_2[1], t_val=val_2[2],
                X_test=test_2[0], y_test=test_2[1], t_test=test_2[2],
                feature_names=feature_names,
                channels=channels
                )
        
        print("\n|-----------------------|")

        del train_2, val_2, test_2, feature_names, channels
        gc.collect()
        
    print(f"\nFinish elaborating {p_name}")

print("\nSplitting completed.")

Subjects:   0%|          | 0/4 [00:00<?, ?it/s]


Elaborating chb04


Labels (chb04):   0%|          | 0/4 [00:00<?, ?it/s]


|--- Splitting task 1 ---|
Set        | Size       | Class 0    | Class 1   
--------------------------------------------------
Train      | 4030       | 4030       | 0         
Val        | 863        | 863        | 0         
Test       | 865        | 865        | 0         

|-----------------------|

|--- Splitting task 2 ---|
Set        | Size       | Class 0    | Class 1   
--------------------------------------------------
Train      | 4030       | 4030       | 0         
Val        | 863        | 863        | 0         
Test       | 865        | 865        | 0         

|-----------------------|

|--- Splitting task 1 ---|
Set        | Size       | Class 0    | Class 1   
--------------------------------------------------
Train      | 4030       | 4030       | 0         
Val        | 863        | 863        | 0         
Test       | 865        | 865        | 0         

|-----------------------|

|--- Splitting task 2 ---|
Set        | Size       | Class 0    | Class 1   
----

Labels (chb12):   0%|          | 0/10 [00:00<?, ?it/s]


|--- Splitting task 1 ---|
Set        | Size       | Class 0    | Class 1   
--------------------------------------------------
Train      | 52         | 26         | 26        
Val        | 216        | 216        | 0         
Test       | 217        | 203        | 14        

|-----------------------|

|--- Splitting task 2 ---|
Set        | Size       | Class 0    | Class 1   
--------------------------------------------------
Train      | 434        | 217        | 217       
Val        | 180        | 90         | 90        
Test       | 211        | 84         | 127       

|-----------------------|

|--- Splitting task 1 ---|
Set        | Size       | Class 0    | Class 1   
--------------------------------------------------
Train      | 56         | 28         | 28        
Val        | 24         | 12         | 12        
Test       | 217        | 217        | 0         

|-----------------------|

|--- Splitting task 2 ---|
Set        | Size       | Class 0    | Class 1   
----

Labels (chb13): 0it [00:00, ?it/s]


Finish elaborating chb13

Elaborating chb19


Labels (chb19): 0it [00:00, ?it/s]


Finish elaborating chb19

Splitting completed.
